# Quantum Fragment Methods Demo

This demonstration showcases three computational approaches for molecular quantum calculations on an alanine molecule. It replicates the procedure from "Molecular Quantum Computation on a Protein":

1. **CCSD for full unfragmented system** - Traditional coupled-cluster calculation
2. **EWF+CCSD for fragments** - Embedded Wave Function with CCSD solver
3. **EWF+(FCI+SQD) for fragments** - Embedded Wave Function with adaptive solvers

We'll compare the total energies and computational approaches for each method.

## Overview

### Method 1: Full System CCSD
- **Pros**: High accuracy reference, no approximations from fragmentation
- **Cons**: Computationally expensive, scales poorly with system size
- **Use case**: Small molecules, benchmark calculations

### Method 2: EWF+CCSD
- **Pros**: More scalable than full system, maintains CCSD accuracy per fragment
- **Cons**: Still uses classical CCSD for all fragments
- **Use case**: Medium-sized systems where classical methods are feasible

### Method 3: EWF+(FCI+SQD)
- **Pros**: Adaptive solver selection, enables quantum computing, exact solutions for small fragments
- **Cons**: Requires quantum hardware/simulation for SQD
- **Use case**: Large systems, hybrid classical-quantum workflows, exploring quantum advantage

### Embedding Benefits
The EWF embedding method enables:
- **Scalability**: Break large systems into manageable fragments
- **Flexibility**: Use different solvers for different fragments
- **Accuracy**: Maintain chemical accuracy through proper bath orbital selection
- **Parallelization**: Solve fragments independently

## Setup and Imports

In [ ]:
from quantum_fragment_methods.workflow import QFWorkflow
from quantum_fragment_methods.application.embedding import EWF
from quantum_fragment_methods.application.solvers.classical_zoo import FCI, CCSD
from quantum_fragment_methods.application.solvers.quantum_zoo.sqd import SQDSolver
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
xyz_file = '/workspace/quantum-fragment-methods/examples/local_demo/system.xyz'

# Read XYZ file content
with open(xyz_file, 'r') as f:
    lines = f.readlines()
    geometry_data = ''.join(lines[2:])  # Skip first two lines (atom count and comment)

print(f"✓ Loaded geometry from: {xyz_file}")
print(f"✓ Number of atoms: {lines[0].strip()}")
print(f"✓ Molecule: {lines[1].strip()}")

## Method 1: CCSD for Full Unfragmented System

First, we perform a traditional CCSD calculation on the entire alanine molecule without fragmentation. This serves as our reference calculation.

**Key Points:**
- No embedding method used
- Single CCSD calculation on the complete system
- Computationally expensive for large systems
- Provides high-accuracy reference energy

In [ ]:
# For full system CCSD, we use PySCF directly (no fragmentation)
import pyscf
from pyscf import cc

print("Running CCSD on full unfragmented system...")

# Create PySCF molecule
mol = pyscf.gto.Mole()
mol.atom = geometry_data
mol.unit = "Angstrom"
mol.basis = "sto-3g"
mol.verbose = 4
mol.build()

In [ ]:
# Run Hartree-Fock
mf = pyscf.scf.RHF(mol)
mf.kernel()
hf_energy = mf.e_tot

In [ ]:
# Run CCSD
mycc = cc.CCSD(mf)
mycc.kernel()
energy_full_ccsd = mycc.e_tot

print(f"\n{'='*60}")
print(f"Method 1: Full System CCSD (No Fragmentation)")
print(f"{'='*60}")
print(f"HF Energy:    {hf_energy:.8f} Ha")
print(f"CCSD Energy:  {energy_full_ccsd:.8f} Ha")
print(f"Correlation:  {(energy_full_ccsd - hf_energy)*1000:.4f} mHa")
print(f"Number of fragments: 1 (unfragmented)")
print(f"{'='*60}\n")

## Method 2: EWF+CCSD for Fragments

Now we use the Embedded Wave Function (EWF) method to fragment the system, then solve each fragment with CCSD.

**Key Points:**
- EWF embedding creates atomic fragments with bath orbitals
- Each fragment solved independently with CCSD
- More scalable than full system calculation
- Maintains high accuracy through embedding

In [ ]:
# Create EWF embedder with configuration
ewf_embedder = EWF(
    bath_type='mp2', 
    truncation=1e-5
    # fragment_type is specified in kernel(), not __init__
)

In [ ]:
# Create workflow with EWF embedder and IAO fragmentation
workflow_ewf_ccsd = QFWorkflow(
    geometry=geometry_data,
    basis='sto-3g',
    embedder=ewf_embedder,
    fragmentation='iao',  # Use IAO fragmentation 
    save_path='results_ewf_ccsd/'
)

# Use CCSD for all fragments
workflow_ewf_ccsd.add_solver_rule(
    solver_factory=lambda frag: CCSD(),
    condition=None,  # Applies to all fragments
    priority=0
)

In [ ]:
# Run workflow
print("Running EWF+CCSD on fragmented system with IAO fragmentation...")
results_ewf_ccsd = workflow_ewf_ccsd.run()
energy_ewf_ccsd = results_ewf_ccsd.total_energy

print(f"\\n{'='*60}")
print(f"Method 2: EWF+CCSD (IAO Fragmentation)")
print(f"{'='*60}")
print(f"Total Energy: {energy_ewf_ccsd:.8f} Ha")
print(f"Number of fragments: {len(results_ewf_ccsd.fragment_energies)}")
print(f"Energy difference from full CCSD: {(energy_ewf_ccsd - energy_full_ccsd)*1000:.4f} mHa")
print(f"{'='*60}\\n")

## Method 3: EWF+(FCI+SQD) for Fragments

Finally, we use EWF embedding with adaptive solver selection based on fragment size:
- **Small fragments** (< 15 orbitals): Full Configuration Interaction (FCI) - exact solution
- **Larger fragments** (≥ 15 orbitals): Sample-based Quantum Diagonalization (SQD) - quantum algorithm

**Key Points:**
- Priority-based solver assignment
- FCI provides exact solutions for small fragments
- SQD enables quantum computing for larger fragments with checkpoint-based workflow
- Handles QPU queue times up to 24 hours
- Demonstrates hybrid classical-quantum approach

**Note on QPU Jobs:**
SQD jobs may be queued for up to 24 hours. The workflow uses checkpoints:
1. First run submits jobs and saves job IDs
2. If jobs are queued, you'll get an error - just re-run later
3. When jobs complete, re-running automatically resumes and completes the workflow

In [ ]:
from typing import Any
import yaml
from pathlib import Path

# Load configuration
config_path = Path('/workspace/quantum-fragment-methods/examples/local_demo/config.yaml')
with open(config_path, 'r') as f:
    config = yaml.safe_load(f)

# Extract configurations
qpu_config = config['qpu']
sqd_config = config['sqd']
embedder_config = config['embedder']['ewf']
workflow_config = config['workflow']

In [ ]:
# Create EWF embedder from config
ewf_embedder_adaptive = EWF(
    bath_type=embedder_config['bath_type'],
    truncation=embedder_config['truncation']
)

In [ ]:
# Create workflow from config
workflow_ewf_adaptive = QFWorkflow(
    geometry=geometry_data,
    basis=workflow_config['basis'],
    embedder=ewf_embedder_adaptive,
    fragmentation=embedder_config['fragmentation'],
    save_path='results_ewf_adaptive/'
)

In [ ]:
# Rule 1: FCI for small fragments (highest priority)
workflow_ewf_adaptive.add_solver_rule(
    solver_factory=lambda frag: FCI(system=frag),
    condition=lambda frag: frag.n_orbitals < 15,
    priority=10
)

In [ ]:
# Rule 2: SQD for larger fragments with QPU backend
from quantum_fragment_methods.qpu import IBMQuantumBackend

# Initialize QPU backend
backend = IBMQuantumBackend(qpu_config)
backend.initialize()
backend.get_backend()

workflow_ewf_adaptive.add_solver_rule(
    solver_factory=lambda frag: SQDSolver(
        backend,
        config=sqd_config
    ),
    condition=None,
    priority=20
)

In [ ]:
# Run workflow with error handling for QPU queueing
from quantum_fragment_methods.workflow import WorkflowResult


print("\nRunning EWF+(FCI+SQD) on fragmented system...")
print("=" * 60)

results_ewf_adaptive: WorkflowResult = workflow_ewf_adaptive.run()

In [ ]:
energy_ewf_adaptive = results_ewf_adaptive.total_energy
    
# Count solver usage
fci_count = sum(1 for frag in results_ewf_adaptive.fragments if frag.n_orbitals < 15)
sqd_count = len(results_ewf_adaptive.fragments) - fci_count
    
print(f"\n{'='*60}")
print(f"Method 3: EWF+(FCI+SQD) - COMPLETED")
print(f"{'='*60}")
print(f"Total Energy: {energy_ewf_adaptive:.8f} Ha")
print(f"Number of fragments: {len(results_ewf_adaptive.fragment_energies)}")
print(f"  - FCI solver used: {fci_count} fragments")
print(f"  - SQD solver used: {sqd_count} fragments")
print(f"Energy difference from full CCSD: {(energy_ewf_adaptive - energy_full_ccsd)*1000:.4f} mHa")
print(f"{'='*60}\n")

## Comparison and Visualization

Let's create a comprehensive comparison of all three methods.

In [ ]:
# Create comparison figure

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
# Plot 1: Total Energies
methods = ['Full CCSD\n(No Fragmentation)', 'EWF+CCSD\n(Fragmented)', 'EWF+(FCI+SQD)\n(Adaptive)']
energies = [energy_full_ccsd, energy_ewf_ccsd, energy_ewf_adaptive]
colors = ['#2E86AB', '#A23B72', '#F18F01']
    
bars = ax1.bar(methods, energies, color=colors, alpha=0.7, edgecolor='black', linewidth=1.5)
ax1.set_ylabel('Total Energy (Ha)', fontsize=12, fontweight='bold')
ax1.set_title('Total Energy Comparison', fontsize=14, fontweight='bold')
ax1.grid(axis='y', alpha=0.3, linestyle='--')
    
# Add value labels on bars
for bar, energy in zip(bars, energies):
    height = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width()/2., height,
         f'{energy:.6f}',
         ha='center', va='bottom', fontsize=10, fontweight='bold')
    
# Plot 2: Energy Differences from Reference
energy_diffs = [
    0.0,  # Reference
    (energy_ewf_ccsd - energy_full_ccsd) * 1000,  # Convert to mHa
    (energy_ewf_adaptive - energy_full_ccsd) * 1000
]
    
bars2 = ax2.bar(methods, energy_diffs, color=colors, alpha=0.7, edgecolor='black', linewidth=1.5)
ax2.set_ylabel('Energy Difference (mHa)', fontsize=12, fontweight='bold')
ax2.set_title('Deviation from Full CCSD Reference', fontsize=14, fontweight='bold')
ax2.axhline(y=0, color='red', linestyle='--', linewidth=2, label='Reference')
ax2.grid(axis='y', alpha=0.3, linestyle='--')
ax2.legend()
    
    # Add value labels
for bar, diff in zip(bars2, energy_diffs):
    height = bar.get_height()
    ax2.text(bar.get_x() + bar.get_width()/2., height,
             f'{diff:.4f}',
             ha='center', va='bottom' if height >= 0 else 'top',
             fontsize=10, fontweight='bold')
    
plt.tight_layout()
plt.savefig('energy_comparison.png', dpi=300, bbox_inches='tight')
plt.show()
    
print("\nFigure saved as 'energy_comparison.png'")

## Summary Table

In [ ]:
import pandas as pd

# Create summary table

summary_data = {
    'Method': [
        'Full CCSD',
        'EWF+CCSD',
        'EWF+(FCI+SQD)'
    ],
    'Embedding': [
        'None',
        'EWF',
        'EWF'
    ],
    'Solver(s)': [
        'CCSD',
        'CCSD',
        'FCI + SQD'
    ],
    'Fragments': [
        1,
        len(results_ewf_ccsd.fragment_energies),
        len(results_ewf_adaptive.fragment_energies)
    ],
    'Total Energy (Ha)': [
        f'{energy_full_ccsd:.8f}',
        f'{energy_ewf_ccsd:.8f}',
        f'{energy_ewf_adaptive:.8f}'
    ],
    'Δ from Reference (mHa)': [
        '0.0000',
        f'{(energy_ewf_ccsd - energy_full_ccsd)*1000:.4f}',
        f'{(energy_ewf_adaptive - energy_full_ccsd)*1000:.4f}'
    ]
}

df = pd.DataFrame(summary_data)
print("\n" + "="*100)
print("SUMMARY: Comparison of Three Computational Methods")
print("="*100)
print(df.to_string(index=False))
print("="*100)

# Save to CSV
df.to_csv('method_comparison.csv', index=False)
print("\nSummary table saved as 'method_comparison.csv'")

## Conclusion

This tutorial demonstrated three approaches to molecular quantum calculations:

1. Traditional full-system CCSD provides a high-accuracy reference
2. EWF+CCSD shows how embedding enables scalable fragmentation
3. EWF+(FCI+SQD) demonstrates adaptive solver selection for hybrid classical-quantum workflows

The quantum fragment methods package provides a flexible framework for exploring these and other computational strategies, making it easy to experiment with different embedding methods and solver combinations while maintaining code clarity and scientific rigor.